# Credit Score / Credit Risk Predictor

Classify applicants as good or bad credit risk based on financial history features.

**Suggested real dataset (Kaggle):** search for `Credit Score Classification Dataset`. Download it and place the CSV as `data/credit_score.csv` next to this notebook to use real data automatically.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
np.random.seed(42)


## 2. Load Data

In [ ]:
import os

DATA_PATH = "data/credit_score.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded real dataset from {DATA_PATH}, shape={df.shape}")
else:
    print("No local dataset found at 'data/credit_score.csv'.")
    print("To use real data: search Kaggle for 'Credit Score Classification Dataset',")
    print("download it, place the CSV at 'credit_score.csv' inside a 'data/' folder next to this notebook,")
    print("and re-run this cell.")
    print("Generating a realistic synthetic dataset instead so the notebook runs end-to-end...\n")

    n = 2000
    income = np.random.normal(55000, 20000, n).clip(10000, 200000)
    debt = np.random.normal(15000, 10000, n).clip(0, 100000)
    payment_history = np.random.uniform(0, 100, n)  # % on-time payments
    credit_utilization = np.random.uniform(0, 100, n)
    num_accounts = np.random.randint(1, 15, n)
    employment = np.random.choice(['employed', 'self_employed', 'unemployed'], n, p=[0.7,0.2,0.1])
    df = pd.DataFrame({
        'annual_income': income,
        'total_debt': debt,
        'payment_history_pct': payment_history,
        'credit_utilization_pct': credit_utilization,
        'num_accounts': num_accounts,
        'employment_status': employment,
    })
    score = (
        df['payment_history_pct'] * 0.4 - df['credit_utilization_pct'] * 0.3
        + (df['annual_income'] / 1000) * 0.1 - (df['total_debt'] / 1000) * 0.2
        + np.where(df['employment_status'] == 'unemployed', -15, 0)
        + np.random.normal(0, 8, n)
    )
    df['credit_risk'] = np.where(score > np.median(score), 'good', 'bad')

    print(f"Synthetic dataset generated, shape={df.shape}")

df.head()


## 3. Exploratory Data Analysis

In [ ]:
print(df.info())
print()
print(df.describe(include='all').T)


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='credit_risk', data=df)
plt.title('Target distribution: credit_risk')
plt.show()

plt.figure(figsize=(10,8))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation heatmap (numeric features)')
plt.show()


## 4. Preprocessing

In [ ]:
df = df.dropna().reset_index(drop=True)

categorical_cols = ['employment_status']
le_dict = {}
for c in categorical_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c].astype(str))
    le_dict[c] = le

X = df.drop(columns=['credit_risk'])
y = df['credit_risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 5. Model Training & Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

results = {}
preds_dict = {}
for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    preds_dict[name] = preds
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='weighted')
    results[name] = {"Accuracy": acc, "F1": f1}
    print(f"{name:20s} | Accuracy: {acc:.3f} | F1: {f1:.3f}")

best_name = max(results, key=lambda k: results[k]["F1"])
best_model = models[best_name]
best_preds = preds_dict[best_name]
print(f"\nBest model: {best_name}")
print()
print(classification_report(y_test, best_preds))


## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_name}')
plt.show()


## 7. Feature Importance

In [ ]:
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.show()

importances


## 8. Conclusion

- Compared a simple linear/logistic baseline against a Random Forest.
- The Random Forest typically captures non-linear relationships better on this kind of tabular data.
- Next steps: hyperparameter tuning (GridSearchCV/RandomizedSearchCV), cross-validation, trying gradient boosting (XGBoost/LightGBM), and more feature engineering once real Kaggle data is loaded.